In [ ]:
-- 실행 컨텍스트 설정
USE ROLE ACCOUNTADMIN;
USE DATABASE DEMO;
USE SCHEMA ML_DEMO;


# Module 5: Inference (추론)

등록된 모델로 배치 추론, Dynamic Table 자동 추론, SPCS 실시간 추론을 수행합니다.

### 추론 방식

| 방식 | API | 적합 사례 |
|------|-----|----------|
| SQL Batch | `mv.run()` | 일별 전체 스코어링 |
| Dynamic Table 체인 | Managed FV → 추론 DT | 자동 갱신 파이프라인 |
| SPCS Real-time | `mv.create_service()` | ms 응답 |


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F
from snowflake.ml.registry import Registry

session = get_active_session()
session.use_database('DEMO')
session.use_schema('ML_DEMO')

reg = Registry(session=session, database_name='DEMO', schema_name='ML_DEMO')
mv = reg.get_model('CUSTOMER_LTV_PREDICTOR').version('V1')
print(f'Model: {mv.model_name} / {mv.version_name}')


## 1. 배치 추론 (`mv.run()`)

Managed FV에서 최신 피처를 조회하여 배치 추론합니다.


In [ ]:
# Managed FV에서 최신 날짜 피처 조회
fv_df = session.table("DEMO.ML_DEMO.CUSTOMER_LTV_FEATURES$1")
latest_date = fv_df.select(F.max("FEATURE_DATE")).collect()[0][0]
inference_df = fv_df.filter(F.col("FEATURE_DATE") == latest_date)
print(f"Inference data: {inference_df.count():,} rows (FEATURE_DATE={latest_date})")

# 배치 추론
predictions = mv.run(inference_df, function_name="predict")
print(f"Predictions: {predictions.count():,} rows")
predictions.select("C_CUSTKEY", "PREDICTED_LABEL").show(10)


In [ ]:
# 예측 결과 저장
predictions.select(
    'C_CUSTKEY', 'C_MKTSEGMENT', 'TOTAL_ORDERS',
    'AVG_ORDER_VALUE', 'TOTAL_REVENUE',
    F.col('PREDICTED_LABEL').alias('PREDICTED_LTV')
).write.mode('overwrite').save_as_table('DEMO.ML_DEMO.CUSTOMER_LTV_SCORES')

print('CUSTOMER_LTV_SCORES saved')
session.table('DEMO.ML_DEMO.CUSTOMER_LTV_SCORES').show(5)


## 2. Dynamic Table 자동 추론

Managed FV를 참조하는 추론 DT를 생성합니다. 피처가 갱신되면 자동으로 재추론됩니다.

```
CUSTOMER_LTV_FEATURES (Managed FV, 1시간 갱신)
    │
    ▼  MODEL!PREDICT()
LIVE_PREDICTED_LTVS (추론 DT, 자동 재추론)
```


In [ ]:
# 추론 Dynamic Table 생성
# Managed FV에서 최신 피처를 읽어 MODEL!PREDICT() 호출
create_dt_sql = """
CREATE OR REPLACE DYNAMIC TABLE DEMO.ML_DEMO.LIVE_PREDICTED_LTVS
    TARGET_LAG = '1 hour'
    WAREHOUSE  = COMPUTE_WH
AS
WITH latest AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY C_CUSTKEY ORDER BY FEATURE_DATE DESC) AS rn
    FROM DEMO.ML_DEMO.CUSTOMER_LTV_FEATURES$1
)
SELECT
    C_CUSTKEY,
    C_MKTSEGMENT,
    TOTAL_ORDERS,
    AVG_ORDER_VALUE,
    TOTAL_REVENUE,
    MODEL(DEMO.ML_DEMO.CUSTOMER_LTV_PREDICTOR)!PREDICT(
        C_MKTSEGMENT, C_ACCTBAL, TOTAL_ORDERS, AVG_ORDER_VALUE,
        TOTAL_REVENUE, AVG_DISCOUNT, AVG_QUANTITY,
        DAYS_SINCE_LAST_ORDER, C_NATIONKEY
    ):PREDICTED_LABEL::FLOAT AS PREDICTED_LTV,
    CURRENT_TIMESTAMP() AS SCORED_AT
FROM latest
WHERE rn = 1
"""

session.sql(create_dt_sql).collect()
print('LIVE_PREDICTED_LTVS created')
session.table('DEMO.ML_DEMO.LIVE_PREDICTED_LTVS').show(5)


## 3. SPCS Real-time Inference Service

모델을 컨테이너 서비스로 배포하여 REST API로 실시간 추론합니다.


In [ ]:
# 기존 서비스 삭제 후 재배포
try:
    mv.delete_service('LTV_INFERENCE_SVC')
except Exception:
    pass

print('Deploying Inference Service...')
mv.create_service(
    service_name='LTV_INFERENCE_SVC',
    service_compute_pool='SYSTEM_COMPUTE_POOL_CPU',
    ingress_enabled=True,
    gpu_requests=None,
)
print('LTV_INFERENCE_SVC deployed')


In [ ]:
# 실시간 추론 테스트
test_df = inference_df.limit(5)
rt_preds = mv.run(test_df, function_name='predict', service_name='LTV_INFERENCE_SVC')
print('Real-time predictions:')
rt_preds.select('C_CUSTKEY', 'PREDICTED_LABEL').show()


In [ ]:
# 서비스 삭제 (비용 관리)
mv.delete_service('LTV_INFERENCE_SVC')
print('Service deleted')


## 정리

| 방식 | 결과물 |
|------|--------|
| Batch | `CUSTOMER_LTV_SCORES` 테이블 |
| Dynamic Table | `LIVE_PREDICTED_LTVS` (자동 갱신) |
| SPCS | REST 엔드포인트 (ms 응답) |

### 다음 단계
Module 6에서 파이프라인 자동화, Module 7에서 모델 모니터링을 수행합니다.
